before this everytime run metadata table

In [0]:
from datetime import datetime
from pyspark.sql.functions import *
import json
from delta.tables import DeltaTable

In [0]:
# Step 1: Read credentials from secret scope
username = dbutils.secrets.get(scope="ecommerce_scope",key="azureSQL-username")
password = dbutils.secrets.get(scope="ecommerce_scope",key="azureSQL-password")

# Step 2: JDBC connection details
jdbc_hostname = "azuresqlserverkviswan8.database.windows.net"
jdbc_port = 1433
jdbc_database = "ecommerce-data-pipeline-db"

jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};database={jdbc_database}"

silver_transformations_rules = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "metadata_silver_config")  # schema.table
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

watermark_metadata = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "watermark_metadata")  # schema.table
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)


In [0]:
metadata_df = silver_transformations_rules.join(watermark_metadata,on="table_name",how="inner")
display(metadata_df)

In [0]:
def getFolderValidPathsByCurrentDate(bronze_path,silver_last_processed):
    folders = dbutils.fs.ls(bronze_path)
    
    current_date_is = datetime.today()
    valid_paths = []

    for folder in folders:
        # path = /bronzelayer/sales/customers/2026-07-22/
        # name = 2026-07-22/
        # folder.name has a trailing slash / : "2026-07-22/"
        # What .strip("/") does
        # Removes / from start & end
        # "2026-07-22/"  →  "2026-07-22"
        folder_name = folder.name.strip("/")
        
        try:
            # folder.name = "2026-07-22/"
            #         ↓ strip("/")
            # folder_name = "2026-07-22"
            #         ↓ strptime()
            # folder_date = datetime(2026, 7, 22)
            #         ↓
            # usable for comparisons
            folder_date = datetime.strptime(folder_name, "%Y-%m-%d")
            if folder_date > silver_last_processed and folder_date <= current_date_is:
                valid_paths.append(folder.path)
        except:
            print(f"Skipping invalid folder: {folder_name}")
    return valid_paths

In [0]:
def applyRulesOnTables(df,rules,primary_key,table_name):

    # -------------------
    # VALIDATIONS
    # -------------------
    
    validations = rules.get("validations", {})
    
    # Checking if table having Primary key validation
    primary_key_validation = validations.get("primary_key_not_null")
    if primary_key_validation:
        if df.filter(col(primary_key).isNull()).limit(1).count() > 0:
            print("There are some Null values in primary keys in Table",table_name)
            print(df.filter(col(primary_key).isNull()))
        df = df.filter(col(primary_key).isNotNull())
    
    # Checking if table having duplications validation
    duplications_validation = validations.get("primary_key_unique")
    if duplications_validation:
        df = df.dropDuplicates([primary_key])
    
    # PhoneNumber Length checking
    phno_length_validation = validations.get("phone_min_length")
    if phno_length_validation:
        phone_no_len_mismatch_rows = df.filter(length(col("phone")) != validations["phone_min_length"])
        if phone_no_len_mismatch_rows.limit(1).count() > 0:
            print("There are some phone numbers with length not equal to 10 in Table",table_name)
            print(phone_no_len_mismatch_rows)
        df = df.filter(length(col("phone")) == validations["phone_min_length"])

    # -------------------
    # Transformations
    # -------------------

    transformations = rules.get("transformations",[])
    
    for tf in transformations:
        column = tf["column"]
        action = tf["action"]

        if action == "to_timestamp":
            df = df.withColumn(column, col(column).cast("timestamp"))
        elif action == "upper":
            df = df.withColumn(column,upper(col(column)))
        elif action == "lower":
            df = df.withColumn(column,lower(col(column)))
        elif action == "capitalize":
            df = df.withColumn(column,initcap(col(column)))
        else:
            print("Skipped Transformarions are :",action,"on columns",column,"In Table",table_name)
    
    # -------------------
    # Derived Column Transformations
    # -------------------

    derived_column_transformations = rules.get("derived_columns",[])

    for dtf in derived_column_transformations:
        name = dtf["name"]
        expression = dtf["expression"]

        df = df.withColumn(name,expr(expression))

    return df

In [0]:
def upsertBronzeWithSilverUsingDelta(table_name, df, key):

    # Add ingestion column
    df = df.withColumn("silver_ingest_date", current_date())

    # Define storage path
    silver_path = f"abfss://silverlayer@ecommercepipelineadls.dfs.core.windows.net/ecommerce_db/{table_name}"

    # Check if table exists (use table_name, NOT path)
    if not DeltaTable.isDeltaTable(spark, silver_path):

        df.write.format("delta") \
            .mode("overwrite") \
            .partitionBy("silver_ingest_date") \
            .option("path", silver_path) \
            .saveAsTable(table_name)

        print(f" Created table {table_name}")

    else:
        delta_table = DeltaTable.forName(spark, table_name)

        delta_table.alias("target").merge(
            df.alias("source"),
            f"target.{key} = source.{key}"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        print(f" Upsert completed for {table_name}")

In [0]:
for row in metadata_df.toLocalIterator():

    table_name = row["table_name"]
    bronze_path = row["bronze_path"]
    silver_table = row["silver_table"]
    file_format = row["file_format"]
    primary_key = row["primary_key"]
    watermark_column = row["watermark_column"]
    rules = json.loads(row["rules"])
    silver_last_processed = row["silver_last_processed"]

    valid_folder_paths = getFolderValidPathsByCurrentDate(
        bronze_path=bronze_path,
        silver_last_processed=silver_last_processed
    )

    if len(valid_folder_paths) == 0:
        print(f"No new folders for {table_name}")
        continue

    df = (
        spark.read.format(file_format)
        .option("inferSchema", True)
        .option("header", True)
        .option("pathGlobFilter", "*.csv")
        .load(valid_folder_paths)
    )

    # Watermark filter
    if silver_last_processed:
        df = df.filter(col(watermark_column) >= lit(silver_last_processed))

    if df.limit(1).count() == 0:
        print(f"No incremental data for {table_name}")
        continue

    # Transform
    transformed_df = applyRulesOnTables(
        df=df,
        rules=rules,
        primary_key=primary_key,
        table_name=table_name
    )

    # Get new watermark AFTER transformation
    new_watermark = transformed_df.select(
        max(watermark_column)
    ).collect()[0][0]

    # Upsert
    upsertBronzeWithSilverUsingDelta(
        df=transformed_df,
        table_name=table_name,
        key=primary_key
    )